# Create Radcliffe Institute Fellowship Awards (FELLOWSHIP PATTERN, Method 5 static-HTML listing scrape)

The **Radcliffe Fellowship** (Harvard Radcliffe Institute for Advanced Study) is a prestigious year-long residential fellowship (~50/year) for scholars, scientists, artists and writers across disciplines. Its named fellows are notable individuals, most already authors in OpenAlex — so this ingest is valuable as an **award<->person linkage** (each row ties a named fellow to the fellowship in a given academic year) and adds a humanities/interdisciplinary coverage angle.

**Method 5 (server-rendered static HTML).** The scraper (`scripts/local/radcliffe_to_s3.py`) iterates the academic-year cohort filter and paginates the public Fellows directory at `radcliffe.harvard.edu/radcliffe-fellowship/fellows?year[]=YYYY-YYYY&page=N` (~25 fellows/page). Each card exposes the fellow's slug and explicit name (`<h3 class="m-card__title" data-text="...">`). The fellowship year comes from the cohort filter (exact). Plain `requests`, no browser.

**Awarding body:** Radcliffe Institute for Advanced Study, Harvard University - F4320332388 (US, DOI 10.13039/100006274; no ROR in OpenAlex)

**Schema choices (FELLOWSHIP pattern - each award is held by a named individual):**
- One row per (fellow, cohort year). `funder_award_id` = `{slug}-{year}` (stable, unique; a returning fellow's distinct cohorts are distinct awards).
- `display_name` = `Radcliffe Fellowship - {name} ({year})`. `funding_type` = `fellowship`.
- `lead_investigator` = the **named fellow** (given/family from the name; `affiliation.name`/`country` NULL - the directory lists no institution, never guessed). `co_lead_investigator`/`investigators` NULL.
- `start_year` = the cohort's first year (real, source-authoritative). `start_date`/`end_date`/`end_year` NULL (academic-year award; no day precision, no published end).
- **Amount/currency NULL with the §6.7 fellowship-pattern waiver** - the Institute does not publish per-fellow stipend amounts in the directory (same waiver as HHMI #44 / CIFAR #79 / Pew #97). Never imputed.
- `funder_scheme` NULL; `description` NULL (no per-fellow citation in the directory).

**S3 location:** `s3a://openalex-ingest/awards/radcliffe/radcliffe_fellows.parquet`

**Provenance:** `radcliffe_fellowship` (verified count=0 on production 2026-06-01).


## Step 1: Create staging table from S3


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.radcliffe_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/radcliffe/radcliffe_fellows.parquet`;


In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.radcliffe_raw;


## Step 1.5: Inspect raw + cohort/coverage

Expected: ~1,407 fellow-cohort rows, 2000-2026. fellow_name / funder_award_id / family_name / start_year ~100%.


In [ ]:
%sql
DESCRIBE openalex.awards.radcliffe_raw;


In [ ]:
%sql
SELECT * FROM openalex.awards.radcliffe_raw LIMIT 5;


In [ ]:
%sql
SELECT start_year, COUNT(*) AS fellows, COUNT(family_name) AS has_family
FROM openalex.awards.radcliffe_raw
GROUP BY start_year
ORDER BY start_year DESC;


In [ ]:
%sql
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT funder_award_id) AS distinct_ids,
       COUNT(DISTINCT slug) AS distinct_people
FROM openalex.awards.radcliffe_raw;


## Step 1.6: Fail-fast - verify Radcliffe funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320332388;


## Step 2: Transform to award schema

FELLOWSHIP pattern: `lead_investigator` is the named fellow (given/family from the name, affiliation NULL - no institution in the source). amount/currency NULL (§6.7 fellowship waiver). funder_scheme NULL.


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.radcliffe_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320332388  -- Radcliffe Institute for Advanced Study, Harvard University
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT('Radcliffe Fellowship - ', n.fellow_name,
           CASE WHEN n.start_year IS NOT NULL THEN CONCAT(' (', n.start_year, ')') ELSE '' END) AS display_name,
    CAST(NULL AS STRING) AS description,
    f.funder_id,
    n.funder_award_id,
    CAST(NULL AS DOUBLE) AS amount,
    CAST(NULL AS STRING) AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'fellowship' AS funding_type,
    CAST(NULL AS STRING) AS funder_scheme,
    'radcliffe_fellowship' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.start_year AS INT) AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN n.family_name IS NOT NULL OR n.given_name IS NOT NULL THEN struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                CAST(NULL AS STRING) AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.radcliffe_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL;


## Step 3: Insert into openalex_awards_raw at priority 165


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'radcliffe_fellowship' AND priority = 165;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    165 AS priority  -- Radcliffe Institute Fellowship
FROM openalex.awards.radcliffe_awards;


## Step 6: Verification

§6.7 amount waiver **applies** (fellowship - no per-fellow stipend published; amount/currency NULL by design). Completeness: display_name / lead_investigator / start_year ~100%.


In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM openalex.awards.radcliffe_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.radcliffe_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(lead_investigator) AS has_lead,
    COUNT(start_year) AS has_start_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(lead_investigator), COUNT(*)) * 100.0, 1) AS pct_lead,
    ROUND(try_divide(COUNT(start_year), COUNT(*)) * 100.0, 1) AS pct_start_year
FROM openalex.awards.radcliffe_awards;


In [ ]:
%sql
SELECT MIN(start_year) AS first_year, MAX(start_year) AS last_year,
       COUNT(*) AS total, COUNT(DISTINCT id) AS distinct_ids
FROM openalex.awards.radcliffe_awards;


In [ ]:
%sql
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.radcliffe_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;


In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 55) AS title, funding_type, start_year,
       lead_investigator.given_name AS given, lead_investigator.family_name AS family
FROM openalex.awards.radcliffe_awards
ORDER BY start_year DESC
LIMIT 10;
